## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-13-21-52-58 +0000,wapo,"FAA grounds flights at all U.S. airports, citi...",https://www.washingtonpost.com/transportation/...
1,2026-11-13-21-50-12 +0000,nyt,Eisenkot Accuses Netanyahu of Fueling ‘Racist’...,https://www.nytimes.com/2026/11/13/world/middl...
2,2026-11-13-21-48-53 +0000,nypost,1 in 5 NYC transit riders say they can’t affor...,https://nypost.com/2026/11/13/us-news/1-in-5-n...
3,2026-11-13-21-48-48 +0000,nyt,Search and Rescue Underway in New Orleans Afte...,https://www.nytimes.com/2026/11/13/us/new-orle...
4,2026-11-13-21-47-57 +0000,startribune,Flights are halted at airports across the U.S....,https://www.startribune.com/flights-are-halted...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2435/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
99,nvidia,34
28,new,27
438,huang,26
101,jensen,22
246,trump,19


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
350,2026-11-13-00-44-00 +0000,wsj,Some of President Trump’s top advisers have st...,https://www.wsj.com/politics/policy/lutnick-fi...,141
175,2026-11-13-16-01-00 +0000,wsj,Treasury Secretary Scott Bessent said Howard L...,https://www.wsj.com/politics/policy/bessent-sa...,134
318,2026-11-13-03-57-01 +0000,bbc,Ex-Nvidia CEO Jensen Huang summoned to testify...,https://www.bbc.co.uk/news/articles/cmq5vd1wdg...,122
283,2026-11-13-10-00-00 +0000,wapo,How disgraced Nvidia CEO Jensen Huang became a...,https://www.washingtonpost.com/technology/inte...,115
237,2026-11-13-12-00-00 +0000,wsj,Congressional Democrats have spun their Nvidia...,https://www.wsj.com/politics/howard-lutnick-ne...,112


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
350,141,2026-11-13-00-44-00 +0000,wsj,Some of President Trump’s top advisers have st...,https://www.wsj.com/politics/policy/lutnick-fi...
373,99,2026-11-12-23-05-02 +0000,missionlocal,"Richmond Buzz: New brunch, new pizza, and a ne...",https://missionlocal.org/2026/11/richmond-buzz...
58,96,2026-11-13-20-29-03 +0000,wapo,"Before he was Pope Leo, he was their bishop. N...",https://www.washingtonpost.com/world/2026/11/1...
322,73,2026-11-13-03-27-00 +0000,wsj,"Sen. Chris Van Hollen (D., Md.) said he would ...",https://www.wsj.com/politics/policy/chris-van-...
318,69,2026-11-13-03-57-01 +0000,bbc,Ex-Nvidia CEO Jensen Huang summoned to testify...,https://www.bbc.co.uk/news/articles/cmq5vd1wdg...
71,58,2026-11-13-20-07-00 +0000,wsj,Rep. Lauren Boebert lost her deep-red Colorado...,https://www.wsj.com/politics/elections/lauren-...
76,50,2026-11-13-20-01-22 +0000,nypost,NYC approved construction plans near site of d...,https://nypost.com/2026/11/13/us-news/nyc-okd-...
113,49,2026-11-13-18-36-57 +0000,bbc,Why is Microsoft's outage lasting so long and ...,https://www.bbc.co.uk/news/articles/cwvg238zrn...
218,48,2026-11-13-13-19-19 +0000,chitri,"Ukraine fires nearly 1,200 drones at Russia as...",https://www.chicagotribune.com/2026/11/13/ukra...
43,43,2026-11-13-21-01-00 +0000,wsj,Of all the ways Chairman Kevin Warsh wants to ...,https://www.wsj.com/economy/central-banking/fe...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
